# Step 1: 데이터 준비

딥페이크 탐지 실습을 위한 데이터를 준비합니다.

## 실습 목표
- 딥페이크 탐지용 데이터셋 구조 이해
- 사전 준비된 Workshop 데이터 다운로드
- S3 버킷에 데이터 업로드
- config.json 생성

## 데이터 구성
- **Real**: 실제 얼굴 이미지
- **Fake**: AI로 생성된 가짜 얼굴 이미지 (StyleGAN)
- **용도**: Fine-tuning 전/후 성능 비교

## 1.1 환경 설정

In [ ]:
import os
import boto3
import sagemaker
from pathlib import Path

# 프로젝트 루트 경로 설정
PROJECT_ROOT = Path(os.getcwd()).parent
print(f"Project Root: {PROJECT_ROOT}")

# SageMaker 세션 설정
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name

# S3 버킷 설정
bucket = sagemaker_session.default_bucket()
prefix = 'deepfake-detection'

print(f"Region: {region}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")

## 1.2 Workshop 데이터 다운로드

강사가 미리 준비한 딥페이크 샘플 데이터를 다운로드합니다.

**데이터 구성:**
| 구분 | Real | Fake | 합계 |
|------|------|------|------|
| Train | 1,000장 | 1,000장 | 2,000장 |
| Validation | 200장 | 200장 | 400장 |
| Test | 200장 | 200장 | 400장 |

> 💡 Test 데이터는 Fine-tuning 전/후 성능 비교에 사용됩니다.

In [ ]:
from pathlib import Path

# 데이터 디렉토리 설정
data_dir = Path('./data')
data_dir.mkdir(exist_ok=True)

# Workshop 데이터 소스 (강사가 준비한 Public S3)
DATA_SOURCE = "s3://deepfake-detection-workshop-public/sample-data"

print(f"데이터 소스: {DATA_SOURCE}")
print("데이터 다운로드 중... (약 1-2분 소요)\n")

# S3에서 데이터 다운로드 (Public bucket - 인증 불필요)
!aws s3 cp {DATA_SOURCE}/ ./data/ --recursive --no-sign-request

print("\n✅ 다운로드 완료!")

In [ ]:
# 데이터 구조 확인
print("데이터 디렉토리 구조:")
!find ./data -type d

print("\n데이터 개수 확인:")
total = 0
for split in ['train', 'val', 'test']:
    split_dir = data_dir / split
    if split_dir.exists():
        real_count = len(list((split_dir / 'real').glob('*')))
        fake_count = len(list((split_dir / 'fake').glob('*')))
        print(f"{split}: Real={real_count}, Fake={fake_count}, Total={real_count+fake_count}")
        total += real_count + fake_count

print(f"\n총 이미지 수: {total}장")

## 1.3 샘플 이미지 확인

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, label in enumerate(['real', 'fake']):
    label_dir = data_dir / 'test' / label
    if label_dir.exists():
        images = list(label_dir.glob('*.jpg')) + list(label_dir.glob('*.png'))
        samples = random.sample(images, min(4, len(images)))
        
        for j, img_path in enumerate(samples):
            img = Image.open(img_path)
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
            axes[i, j].set_title(f"{label.upper()}")

plt.suptitle("샘플 이미지 (Real vs Fake)", fontsize=14)
plt.tight_layout()
plt.show()

## 1.4 내 S3 버킷에 업로드

다운로드한 데이터를 본인의 S3 버킷에 업로드합니다.

In [ ]:
# S3에 데이터 업로드
print(f"내 S3 버킷에 업로드 중: s3://{bucket}/{prefix}/data/")

s3_data_path = sagemaker_session.upload_data(
    path='./data',
    bucket=bucket,
    key_prefix=f'{prefix}/data'
)

print(f"\n✅ 업로드 완료: {s3_data_path}")

## 1.5 설정 저장

다음 노트북에서 사용할 설정을 저장합니다.

In [ ]:
import json

config = {
    'bucket': bucket,
    'prefix': prefix,
    'project_root': str(PROJECT_ROOT),
    's3_data_path': s3_data_path,
    's3_train_path': f's3://{bucket}/{prefix}/data/train',
    's3_val_path': f's3://{bucket}/{prefix}/data/val',
    's3_test_path': f's3://{bucket}/{prefix}/data/test',
    'local_test_path': str(data_dir.absolute() / 'test'),
    'role': role,
    'region': region
}

# 프로젝트 루트에 config.json 저장
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ 설정 저장 완료: {config_path}")
print("\n저장된 설정:")
print(json.dumps(config, indent=2))

## ✅ 완료!

데이터 준비가 완료되었습니다.

**확인 사항:**
- [x] 딥페이크 데이터 다운로드 완료
- [x] Train/Val/Test 데이터 확인
- [x] 내 S3 버킷에 업로드 완료
- [x] config.json 저장 완료

---

**➡️ 다음 단계: `2_before_evaluation/evaluate_before.ipynb`**

Fine-tuning 전 모델의 성능을 평가합니다.